In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class MatrixDataset(Dataset):
    def __init__(self, matrices, labels):
        self.matrices = matrices  # numpy array/list: [N, H, W]
        self.labels = labels      # numpy array/list: [N]

    def __len__(self):
        return len(self.matrices)

    def __getitem__(self, idx):
        mat = torch.tensor(self.matrices[idx], dtype=torch.float32)
        mat = (mat - mat.mean()) / (mat.std() + 1e-6)  # normalize
        mat = mat.unsqueeze(0)  # [1, H, W] for CNN
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mat, label

In [3]:
class CNN(nn.Module):
    def __init__(self, num_classes, input_shape):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Figure out flattened size by running dummy input
        with torch.no_grad():
            dummy = torch.zeros(1, 1, *input_shape)  # (B, C, H, W)
            out = self._forward_features(dummy)
            flat_size = out.view(1, -1).size(1)

        self.fc1 = nn.Linear(flat_size, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def _forward_features(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._forward_features(x)
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
train = np.loadtxt("../fiot_highway2-main/train.txt", dtype=str).tolist() #[ ['data/filenum', 'classification 1-9'], ... ]
test  = np.loadtxt("../fiot_highway2-main/test.txt",  dtype=str).tolist() #[ ['data/filenum', 'classification 1-9'], ... ]

# Split into paths and classes
train_file_paths = [row[0] for row in train]
test_file_paths  = [row[0] for row in test]

train_labels     = [int(row[1]) for row in train]
test_labels      = [int(row[1]) for row in test]

train_matrixes  = [np.load("../fiot_highway2-main/" + fp) for fp in train_file_paths]
test_matrixes   = [np.load("../fiot_highway2-main/" + fp) for fp in test_file_paths]

H, W = 512, 243
num_classes = 9 #0-8


In [ ]:
## Attempt to use to classify data

train_dataset = MatrixDataset(train_matrixes[:int(len(train_matrixes) * 0.25)], train_labels)
test_dataset  = MatrixDataset(test_matrixes[:int(len(train_matrixes) * 0.25)], test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
# -----------------------------
# Training
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(num_classes=num_classes, input_shape=(H, W)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    model.train()
    running_loss = 0.0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        # Forward pass
        outputs = model(X)
        loss = criterion(outputs, y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader)

    # -----------------------------
    # Evaluation
    # -----------------------------
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            _, preds = torch.max(outputs, 1)

            correct += (preds == y).sum().item()
            total += y.size(0)
    
    acc = correct / total * 100
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Test Accuracy: {acc:.2f}%")